In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, current_timestamp, input_file_name, from_json,get_json_object
from pyspark.sql.types import ArrayType, StringType, MapType
import os


In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
raw_input_path = "../raw_data/"
bronze_output_path = "../data_lake/bronze/batch_data/"
bronze_hospital_path = "../data_lake/bronze/hospital_data/"
bronze_practitioner_path = "../data_lake/bronze/practitioner_data/"
hospital_paths = []
practitioner_paths = []

In [ ]:
def process_batch_data(input_files, output_path):

    raw_df = (
        spark.read
        .format("binaryFile")
        .load(input_files)
        .select( col("content").cast("string").alias("raw_json"))
        .withColumn("input_file_name", input_file_name())
    )

    result_df = raw_df.select(
        "raw_json",
        get_json_object("raw_json", "$.resourceType").alias("bundle_resource_type"),
        get_json_object("raw_json", "$.type").alias("bundle_type"),
        get_json_object("raw_json", "$.entry").alias("entry_raw_json"),
        "input_file_name"
    )

    intermediate_df_1 = result_df.select(
        "bundle_resource_type",
        "bundle_type",
        from_json("entry_raw_json", ArrayType(MapType(StringType(), StringType()))).alias("entry_array"),
        "input_file_name"
    ).withColumn("entries", explode("entry_array")).select(
        "bundle_resource_type",
        "bundle_type",
        col("entries"),
        "input_file_name"
    )

    final_df = (intermediate_df_1.select(
        "bundle_resource_type",
        "bundle_type",
        col("entries.fullUrl").alias("fullUrl"),
        from_json(col("entries.resource"), MapType(StringType(), StringType())).alias("resource"),
        from_json(col("entries.request"), MapType(StringType(), StringType())).alias("request"),
        "input_file_name"
    ).withColumn("resource_type", col("request").getItem("url")) # resource type can be easily extracted from request url, instead of parsing complex resource json. I spent lots of time trying to parse it from resource json.
        .withColumn("ingestion_timestamp", current_timestamp())
    )

    final_df.write.mode("append").partitionBy("resource_type").parquet(output_path)

In [ ]:
folders =  os.listdir(raw_input_path)

In [ ]:
for folder in folders:
    if folder.startswith("folder_"):
        path = raw_input_path+folder
        all_files = os.listdir(path)
        files = [path + "/"+file for file in all_files if not (file.startswith("hospital") or file.startswith("practitioner"))]
        
        hospital_paths += [path + "/"+file for file in all_files if file.startswith("hospital")]
        practitioner_paths += [path + "/"+file for file in all_files if file.startswith("practitioner")]
    
        # process_batch_data(files, bronze_output_path)

In [ ]:
if len(hospital_paths) !=0:    
    process_batch_data(hospital_paths, bronze_hospital_path )

In [ ]:
if len(practitioner_paths) !=0:    
    process_batch_data(practitioner_paths, bronze_practitioner_path )

In [ ]:
spark.stop()